In [3]:
import cv2
import numpy as np
import os
from scipy.signal import butter, sosfiltfilt, sosfilt
from scipy import fftpack
from tqdm.notebook import tqdm
from pathlib import Path

def filter_breathing_fast(input_path, output_dir, bw=0.05):
    """
    Remove ventilator breathing oscillation from a fixed-camera ICG video.
    Uses vectorized SOS band-stop filtering for speed.

    Parameters:
        input_path (str): Path to input video
        output_dir (str): Directory to save filtered video
        bw (float): Bandwidth around detected breathing frequency in Hz
    """

    os.makedirs(output_dir, exist_ok=True)

    # --- create output filename ---
    base = os.path.basename(input_path)
    name, ext = os.path.splitext(base)
    output_path = os.path.join(output_dir, f"{name}_filtered{ext}")

    # --- load video ---
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {input_path}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fps = cap.get(cv2.CAP_PROP_FPS)

    print(f"Frames: {frame_count}, Resolution: {width}x{height}, FPS: {fps}")

    # Read all frames into memory as grayscale
    video = np.zeros((frame_count, height, width), dtype=np.float32)
    for i in tqdm(range(frame_count), desc="Reading frames"):
        ret, frame = cap.read()
        if not ret:
            video = video[:i]
            frame_count = i
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        video[i] = gray.astype(np.float32)
    cap.release()

    # --- detect ventilator frequency ---
    mean_signal = video.mean(axis=(1,2))
    x = np.arange(frame_count)
    trend = np.polyval(np.polyfit(x, mean_signal, 1), x)
    detrended = mean_signal - trend

    window = np.hanning(frame_count)
    fft_vals = fftpack.fft(detrended * window)
    freqs = fftpack.fftfreq(frame_count, 1 / fps)

    pos = freqs > 0
    freqs_pos = freqs[pos]
    fft_pos = np.abs(fft_vals[pos])

    vent_band = (freqs_pos > 0.18) & (freqs_pos < 0.35)
    idx = np.argmax(fft_pos[vent_band])
    breath_freq = freqs_pos[vent_band][idx]

    print(f"Detected ventilator frequency: {breath_freq:.3f} Hz ({breath_freq*60:.1f} bpm)")

    # --- SOS bandstop filter design ---
    low = max(0, breath_freq - bw)
    high = breath_freq + bw
    sos = butter(N=2, Wn=[low, high], btype='bandstop', fs=fps, output='sos')

    # --- apply filter vectorized ---
    flat = video.reshape(frame_count, -1)  # shape: (frames, pixels)
    filtered_flat = sosfilt(sos, flat, axis=0)
    filtered_video = filtered_flat.reshape(frame_count, height, width)
    filtered_video = np.clip(filtered_video, 0, 255).astype(np.uint8)

    # --- save filtered video ---
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height), False)
    for i in tqdm(range(frame_count), desc="Writing filtered video"):
        writer.write(filtered_video[i])
    writer.release()

    print(f"Filtered video saved to: {output_path}")
    return output_path

In [2]:
# Folder containing your videos
input_folder = "/home/jovyan/M2-2_stage/ICG_video's/"
output_folder = "/home/jovyan/M2-2_stage/ICG_video's_filtered/"

# Iterate over all .mp4 files in the folder
for file_path in Path(input_folder).glob("*.mp4"):
    print(f"Processing: {file_path.name}")
    filter_breathing_fast(str(file_path), output_folder)

Processing: FIM001_ICG.mp4
Frames: 6304, Resolution: 1280x720, FPS: 30.0


Reading frames: 100%|██████████| 6304/6304 [00:19<00:00, 330.90it/s]


Detected ventilator frequency: 0.266 Hz (16.0 bpm)


Writing filtered video: 100%|██████████| 6304/6304 [00:54<00:00, 114.71it/s]


Filtered video saved to: /home/jovyan/M2-2_stage/ICG_video's_filtered/FIM001_ICG_filtered.mp4
Processing: FIM002_ICG.mp4
Frames: 4191, Resolution: 1280x720, FPS: 30.0


...

Filtered video saved to: /home/jovyan/M2-2_stage/ICG_video's_filtered/FIM016_ICG_filtered.mp4
Processing: FIM017_ICG.mp4
Frames: 3545, Resolution: 1280x720, FPS: 30.0


Reading frames: 100%|██████████| 3545/3545 [00:11<00:00, 297.54it/s]


Detected ventilator frequency: 0.203 Hz (12.2 bpm)


Writing filtered video: 100%|██████████| 3545/3545 [00:31<00:00, 111.16it/s]


Filtered video saved to: /home/jovyan/M2-2_stage/ICG_video's_filtered/FIM017_ICG_filtered.mp4


In [4]:
input_video = "/home/jovyan/M2-2_stage/ICG_video's/FIM018_ICG.mp4"
output_dir = "/home/jovyan/M2-2_stage/ICG_video's_filtered/"

filter_breathing_fast(input_video, output_dir)

Frames: 3630, Resolution: 1280x720, FPS: 30.0


Reading frames:   0%|          | 0/3630 [00:00<?, ?it/s]

Detected ventilator frequency: 0.231 Hz (13.9 bpm)


Writing filtered video:   0%|          | 0/3630 [00:00<?, ?it/s]

Filtered video saved to: /home/jovyan/M2-2_stage/ICG_video's_filtered/FIM018_ICG_filtered.mp4


"/home/jovyan/M2-2_stage/ICG_video's_filtered/FIM018_ICG_filtered.mp4"